Step 1: Import lib

In [1]:
import argparse
import json
import os
import random
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.optim import SGD
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from tqdm import tqdm

Step 2: Config

In [2]:
# DATASET_ROOT = "/Users/tranminhhuy/Documents/dataset" # update dataset path
DATASET_ROOT = r"D:\dataset"
MODEL_NAME = "resnext"   # "resnext", "wideresnet", "efficientnet"
EPOCHS = 1 # look back train set
BATCH_SIZE = 8
IMG_SIZE = 224
LR = 0.01 # learning rate -> based on original paper
MOMENTUM = 0.9 # based on original paper
TRAIN_RATIO = 0.7 # percentage dataset used to train
SEED = 42 # for random
NUM_WORKERS = 0 # worker helps to load data
PRETRAINED = False # model does not use data learnt before

OUTPUT_DIR = Path("./outputs_notebook")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Step 3: Seed & device #local mac

In [3]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

set_seed(SEED)
device = get_device()
print("Using device:", device)

Using device: mps


Step 3: Seed & device local win

In [3]:
import torch
import random
import numpy as np

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

set_seed(SEED)
device = get_device()

print("Using device:", device)
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

Using device: cuda
CUDA available: True
GPU name: NVIDIA GeForce RTX 3060


Step 4: Transforms

In [4]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

Step 5: Load dataset

In [5]:
base_dataset = datasets.ImageFolder(DATASET_ROOT)
train_dataset_full = datasets.ImageFolder(DATASET_ROOT, transform=train_transform)
test_dataset_full = datasets.ImageFolder(DATASET_ROOT, transform=test_transform)

print("Classes:", base_dataset.classes)
print("Num classes:", len(base_dataset.classes))
print("Total images:", len(base_dataset))

FileNotFoundError: Found no valid file for the classes wildlife. Supported extensions are: .jpg, .jpeg, .png, .ppm, .bmp, .pgm, .tif, .tiff, .webp

Step 6: Split dataset with 70% train and 30% test

In [6]:
from collections import defaultdict
import random

def stratified_split_indices(targets, train_ratio=0.7, seed=42):
    rng = random.Random(seed)
    class_to_indices = defaultdict(list)

    for idx, label in enumerate(targets):
        class_to_indices[label].append(idx)

    train_indices = []
    test_indices = []

    for label, indices in class_to_indices.items():
        indices = indices[:]
        rng.shuffle(indices)
        split_idx = int(len(indices) * train_ratio)

        train_indices.extend(indices[:split_idx])
        test_indices.extend(indices[split_idx:])

    rng.shuffle(train_indices)
    rng.shuffle(test_indices)
    return train_indices, test_indices


train_indices, test_indices = stratified_split_indices(
    base_dataset.targets,
    train_ratio=TRAIN_RATIO,
    seed=SEED
)

train_dataset = Subset(train_dataset_full, train_indices)
test_dataset = Subset(test_dataset_full, test_indices)

print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

Train samples: 70
Test samples: 30


Step 7: Dataloader prepare data so that model can read it in batches

In [7]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

print("Train loader batches:", len(train_loader))
print("Test loader batches:", len(test_loader))

Train loader batches: 9
Test loader batches: 4


Step 8: Build model: create model based on the config above

In [8]:
def build_model(model_name: str, num_classes: int, pretrained: bool = False):
    model_name = model_name.lower()

    if model_name == "resnext":
        weights = models.ResNeXt50_32X4D_Weights.DEFAULT if pretrained else None
        model = models.resnext50_32x4d(weights=weights)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
        return model

    raise ValueError(f"Unsupported model: {model_name}")


model = build_model(
    model_name=MODEL_NAME,
    num_classes=len(base_dataset.classes),
    pretrained=PRETRAINED
).to(device)

criterion = nn.CrossEntropyLoss() # loss function
optimizer = SGD(model.parameters(), lr=LR, momentum=MOMENTUM) # optimizer

print("Model name:", MODEL_NAME)
print("Num classes:", len(base_dataset.classes))
print("Device:", device)

Model name: resnext
Num classes: 10
Device: mps


Step 9: Create Train and evaluate function

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device): #train_loader teach model
    model.train()
    running_loss = 0.0
    correct = 0 # the number of right prediction
    total = 0 # total images go through model

    for images, labels in tqdm(loader, desc="Training", leave=False): 
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device): #test_loader check model after learning
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc="Evaluating", leave=False):
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

Step 10: Start train model 

In [10]:
history = []
best_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Test  Loss: {test_loss:.4f} | Test  Acc: {test_acc:.4f}")

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
    })

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "classes": base_dataset.classes,
                "model_name": MODEL_NAME,
                "img_size": IMG_SIZE,
            },
            OUTPUT_DIR / f"best_{MODEL_NAME}.pth"
        )

print("\nBest test accuracy:", best_acc)


Epoch 1/1


Train Loss: 10.9501 | Train Acc: 0.0714
Test  Loss: 30.8843 | Test  Acc: 0.1000

Best test accuracy: 0.1
